In [1]:
import time

t0 = time.time()
from hypernetx import Hypergraph
print("First import:", round(time.time() - t0, 2), "s")

t1 = time.time()
from hypernetx import Hypergraph
print("Second import:", round(time.time() - t1, 4), "s")

First import: 4.47 s
Second import: 0.0 s


In [2]:
# Run-once cell: imports + small helpers
from hypernetx import Hypergraph

def incidence_as_sets(H: Hypergraph):
    """Return {edge: set(nodes)} for easy reading/logic."""
    return {e: set(nodes) for e, nodes in H.incidence_dict.items()}

def print_edges(H: Hypergraph, title="Edges"):
    """Pretty-print hyperedges."""
    print(title)
    for e, nodes in incidence_as_sets(H).items():
        print(f"  {e}: {sorted(nodes)}")

In [3]:
def remove_contained_edges(H: Hypergraph) -> Hypergraph:
    """
    Keep only maximal hyperedges (toplexes).
    This removes any edge that is contained in another edge.
    """
    maximal_edges = H.toplexes()
    return H.restrict_to_edges(maximal_edges)

In [5]:
import pandas as pd
from hypernetx import Hypergraph

E = {
    "e1": {"a","b","c","d"},
    "e2": {"a","b"},
    "e3": {"c","d","e"},
    "e4": {"d","e"},
}

# Build an incidence table: one row per (edge, node) membership
rows = []
for e, nodes in E.items():
    for v in nodes:
        rows.append({
            "edges": e,
            "nodes": v,
            "weight": 1,              # safe default
            "misc_properties": None   # <-- important: include this column
        })

inc_df = pd.DataFrame(rows)

H = Hypergraph(inc_df)  # this path is usually rock-solid
H

None hypernetx.classes.hypergraph.Hypergraph

In [6]:
def print_edges(H, title="Edges"):
    print(title)
    for e, nodes in H.incidence_dict.items():
        print(f"  {e}: {sorted(nodes)}")

print_edges(H, "Original")

Original
  e1: ['a', 'b', 'c', 'd']
  e2: ['a', 'b']
  e3: ['c', 'd', 'e']
  e4: ['d', 'e']


In [7]:
def remove_contained_edges(H):
    return H.restrict_to_edges(H.toplexes())

Hmin = remove_contained_edges(H)
print_edges(Hmin, "\nMinimized (toplexes only)")

Hd = Hmin.dual()
print_edges(Hd, "\nDual of minimized")


Minimized (toplexes only)
  e1: ['a', 'b', 'c', 'd']
  e3: ['c', 'd', 'e']

Dual of minimized
  a: ['e1']
  b: ['e1']
  c: ['e1', 'e3']
  d: ['e1', 'e3']
  e: ['e3']
